In [39]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

In [40]:
output_path = "../../output/protenn2/test_arch"
input_data_folder = "../../datasets/v3"
model_path = os.path.join(output_path, "best_model.pt")
label_encoder_path = os.path.join(output_path, "label_encoder.pkl")

In [41]:
import matplotlib.pyplot as plt
import numpy as np
import os # Assuming this is available

# ... (your existing code for model, dataloader, label_encoder, output_dir, device) ...

def visualize_predictions(model, dataloader, label_encoder, output_dir, device, num_visualizations=5):
    """
    Visualizes per-residue predictions for a subset of the validation set,
    showing the *entire padded protein sequence*.
    """
    model.eval()
    os.makedirs(output_dir, exist_ok=True)

    id_to_label = {i: label for i, label in enumerate(label_encoder.classes_)}

    visualized_count = 0

    with torch.no_grad():
        for i, (x, y_true, domain_id) in enumerate(dataloader):
            if visualized_count >= num_visualizations:
                break

            # Handle domain_id batching (if batch_size > 1, domain_id will be a tuple/list)
            # Assuming batch_size is 1 based on your setup, so domain_id[0] is the protein ID string
            current_domain_id_str = domain_id[0] if isinstance(domain_id, (list, tuple, torch.Tensor)) else domain_id

            for k, v in x.items():
                x[k] = v.to(device, non_blocking=True)
            y_true = y_true.to(device, non_blocking=True)

            outputs = model(x)
            outputs = outputs.permute(0, 2, 1)
            y_pred = torch.argmax(outputs, dim=1)

            for batch_idx in range(x["embedding"].shape[0]):
                if visualized_count >= num_visualizations:
                    break

                full_display_length = x["embedding"].shape[1]

                true_labels_tensor = y_true[batch_idx].cpu()
                pred_labels_tensor = y_pred[batch_idx].cpu()

                true_labels_to_plot = true_labels_tensor.numpy()
                pred_labels_to_plot = pred_labels_tensor.numpy()

                true_labels_str = [id_to_label[int(id_val)] for id_val in true_labels_to_plot]
                pred_labels_str = [id_to_label[int(id_val)] for id_val in pred_labels_to_plot]

                # Collect all unique labels to ensure consistent mapping across plots and legend
                all_labels_in_current_plot = sorted(list(set(true_labels_str + pred_labels_str)))
                label_to_plot_value = {label: idx for idx, label in enumerate(all_labels_in_current_plot)}

                true_plot_values = [label_to_plot_value[label] for label in true_labels_str]
                pred_plot_values = [label_to_plot_value[label] for label in pred_labels_str]

                # --- The CRUCIAL CHANGE ---
                # Set vmin and vmax for imshow to cover the full range of integer indices
                # This ensures consistent mapping to colormap colors
                vmin_plot = 0
                vmax_plot = len(all_labels_in_current_plot) - 1 # Max index value

                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
                fig.suptitle(f"Domain {current_domain_id_str} Residue-wise Prediction (Full Sequence)")

                # Plot true labels with consistent vmin/vmax
                ax1.imshow(np.array(true_plot_values).reshape(1, -1), cmap='tab20', aspect='auto',
                           extent=[0, full_display_length, 0, 1],
                           vmin=vmin_plot, vmax=vmax_plot) # <--- Added vmin/vmax
                ax1.set_yticks([])
                ax1.set_title("True CATH Domains")
                ax1.set_ylabel("True")
                ax1.set_xlim(0, full_display_length)

                # Plot predicted labels with consistent vmin/vmax
                ax2.imshow(np.array(pred_plot_values).reshape(1, -1), cmap='tab20', aspect='auto',
                           extent=[0, full_display_length, 0, 1],
                           vmin=vmin_plot, vmax=vmax_plot) # <--- Added vmin/vmax
                ax2.set_yticks([])
                ax2.set_title("Predicted CATH Domains")
                ax2.set_xlabel("Residue Index")
                ax2.set_ylabel("Predicted")
                ax2.set_xlim(0, full_display_length)

                # Create a custom legend using the same normalization logic
                cmap_norm = vmax_plot # Same as len(all_labels_in_current_plot) - 1
                if cmap_norm == 0: cmap_norm = 1 # Avoid division by zero if only one unique label

                handles = [plt.Rectangle((0, 0), 1, 1, color=plt.cm.tab20(label_to_plot_value[label] / cmap_norm))
                           for label in all_labels_in_current_plot]
                ax2.legend(handles, all_labels_in_current_plot, loc='upper center', bbox_to_anchor=(0.5, -0.2),
                           fancybox=True, shadow=True, ncol=3)

                plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                plt.savefig(os.path.join(output_dir, f"{current_domain_id_str}.png"))
                plt.close(fig)

                visualized_count += 1
                print(f"Generated visualization for protein {visualized_count}")

In [42]:
from src.protenn2.model import CathPredEnn2
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.utils import get_train_val_test_paths, calculate_max_protein_length

# Define paths (adjust these to your specific project structure)
# IMPORTANT: Make sure these paths point to where your trained model,
# label encoder, and dataset CSVs/embeddings are located.


visualization_output_dir = os.path.join(output_path, "visualization_results")

# Set device (MPS for Apple Silicon, otherwise CPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU) for visualization.")
else:
    device = torch.device("cpu")
    print("MPS not available, falling back to CPU for visualization.")

# Load LabelEncoder
try:
    with open(label_encoder_path, "rb") as f:
        label_encoder = pickle.load(f)
    print(f"Loaded LabelEncoder with {len(label_encoder.classes_)} classes.")
except FileNotFoundError:
    print(f"Error: LabelEncoder file not found at {label_encoder_path}. Please check the path.")
    exit()  # Exit if the label encoder isn't found

# Get data paths
train_path, val_path, test_path = get_train_val_test_paths(input_data_folder)

# Create validation dataset and dataloader
# Pass the correct embedding_dir to the dataset constructor
val_dataset = CathPredPerResidueDataset(val_path, label_encoder,
                                        embedding_dir="../../data/embeddings/protein_embeddings", fit=False)
max_protein_length = calculate_max_protein_length(input_data_folder)

if max_protein_length == 0:
    print(
        "Error: Max protein length is 0. This usually means no valid protein embeddings were found or data paths are incorrect.")
    print(
        "Please ensure 'input_data_folder' and 'embedding_base_dir' are correctly set and contain the necessary files.")
    exit()

collate_fn = create_protein_collate_fn(max_protein_length, val_dataset.no_domain_encoded_id)
# Use batch_size=1 for easier visualization of individual proteins
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# Initiate and load the model
num_classes = len(label_encoder.classes_)
model = CathPredEnn2(num_classes=num_classes)
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    print("Model loaded successfully.")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_path}. Please check the path.")
    exit()
except Exception as e:
    print(f"Error loading model: {e}. Ensure CathPredEnn2 model definition matches the saved state_dict.")
    exit()

# Perform and visualize predictions
print("\nStarting visualization...")
visualize_predictions(model, val_dataloader, label_encoder, visualization_output_dir, device, num_visualizations=100)
print(f"\nVisualizations saved to {visualization_output_dir}")

Using MPS (Apple Silicon GPU) for visualization.
Loaded LabelEncoder with 804 classes.
Max protein length: 599
Model loaded successfully.

Starting visualization...
Generated visualization for protein 1
Generated visualization for protein 2
Generated visualization for protein 3
Generated visualization for protein 4
Generated visualization for protein 5
Generated visualization for protein 6
Generated visualization for protein 7
Generated visualization for protein 8
Generated visualization for protein 9
Generated visualization for protein 10
Generated visualization for protein 11
Generated visualization for protein 12
Generated visualization for protein 13
Generated visualization for protein 14
Generated visualization for protein 15
Generated visualization for protein 16
Generated visualization for protein 17
Generated visualization for protein 18
Generated visualization for protein 19
Generated visualization for protein 20
Generated visualization for protein 21
Generated visualization f